In [1]:
from qdrant_client import QdrantClient
from qdrant_client import models, QdrantClient
import math
import unicodedata
import re
from underthesea import word_tokenize
from fastembed.sparse import SparseTextEmbedding
from langchain_huggingface import HuggingFaceEmbeddings

DENSE_MODEL_HF = r"E:\QuangNV\QandA_Bank\models\Vietnamese_Embedding"
# client = QdrantClient(url = "http://localhost:6333/")
client = QdrantClient(url = "http://100.65.71.50:6333/")
limit_dense = 5
limit_space = 5

def normalize_vi(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize("NFC", text.lower())
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def tokenize_vi(text: str) -> str:
    text = normalize_vi(text)
    # underthesea trả về dạng: "bác_hồ kính_yêu"
    return word_tokenize(text, format="text")

sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")
dense_model_hf = HuggingFaceEmbeddings(
    model_name=DENSE_MODEL_HF,
    model_kwargs={
        "local_files_only": True,
        "trust_remote_code": True
        })
def embed(request: str):
    dense_vector = dense_model_hf.embed_query(request)
    # sparse_embedding = list(sparse_model.embed([request]))[0] # Returns a SparseEmbedding object
    sparse_text = tokenize_vi(request)
    sparse_embedding = list(sparse_model.embed([sparse_text]))[0]
    indices = sparse_embedding.indices.tolist()
    values = sparse_embedding.values.tolist()
    sparse_query = models.SparseVector(indices=indices, values=values)
    return dense_vector, sparse_query
from langchain_core.documents import Document
def search_qa_bank_done(dense_vector, sparse_query, top_k, collection: str, hybrid: str = "hybrid"):
    # Gom kết quả toàn bộ
    all_results = []
    if hybrid == "hybrid":
        results = client.query_points(
            collection_name=collection,
            query=models.FusionQuery(
                fusion=models.Fusion.RRF # we are using reciprocal rank fusion here
            ),
            prefetch=[
            models.Prefetch(
                query=dense_vector,
                using="dense",
                limit= limit_dense # tên field dense vector trong collection
            ),
            models.Prefetch(
                query=sparse_query,
                using="sparse",
                limit = limit_space # tên field sparse vector trong collection  
            ),
            ],
            query_filter=None, # If you don't want any filters for now
            limit=top_k, # 5 the closest results
        ).points
    elif hybrid == "dense":
        results = client.query_points(
            collection_name=collection,
            query=dense_vector,          # sparse vector (BM25/SPLADE)
            using="dense",              # tên sparse vector field
            limit=top_k
        ).points
    elif hybrid == "sparse":
        results = client.query_points(
            collection_name=collection,
            query=sparse_query,          # sparse vector (BM25/SPLADE)
            using="sparse",              # tên sparse vector field
            limit=limit_space
        ).points
        if results:
            scores = [p.score for p in results]
            min_s, max_s = min(scores), max(scores)
            for p in results:
                if max_s > min_s:
                    p.score = (p.score - min_s) / (max_s - min_s)
                else:
                    p.score = 1.0
            # scores = [p.score for p in results]

            # # tránh overflow
            # max_score = max(scores)
            # exp_scores = [
            #     math.exp((s - max_score) / 1)
            #     for s in scores
            # ]
            # sum_exp = sum(exp_scores)

            # for p, e in zip(results, exp_scores):
            #     p.score = e / sum_exp
                
    for point in results:
    # Lấy payload (nội dung và metadata) từ Qdrant
        payload = point.payload or {}
        content = payload.get("page_content", "")
        metadata = payload.get("metadata", {})
    # Tạo lại Document
        doc = Document(page_content=content, metadata=metadata)
        # Lưu kết quả
        all_results.append({
            "collection": collection,
            "content": doc.page_content,
            "metadata": doc.metadata,
            "score": float(point.score)
        })
    return {"results": all_results}
def normalize(text):
    if not text:
        return ""
    return text.lower().strip()


In [2]:
text = """Bác đã tính về thời gian đến muộn cuộc họp của đồng chí cán bộ như thế nào?"""
text = normalize(text)
a, b = embed(text)
res = search_qa_bank_done(a,b,3,"sd_5_general_v3",hybrid="hybrid")
res

{'results': [{'collection': 'sd_5_general_v3',
   'content': '[Reference:"Sách Sống đẹp theo gương Bác Hồ, Lớp 5 Bài 3"]\n### Khám phá:\n\n## Câu Hỏi:\n- Vì sao đồng chí cấp tướng đến muộn?\n- Bác đã chỉ ra nguyên nhân đến muộn của đồng chí cấp tướng là do đâu?\n- Bác đã tính về thời gian đến muộn cuộc họp của đồng chí cán bộ như thế nào?\n- Bác đã nói thế nào khi có người đề nghị Bác hoãn đến thăm lớp chỉnh huấn của anh chị em trí thức?\n- Qua câu chuyện trên, chúng ta học được bài học gì về tấm gương tôn trọng kỉ luật thời gian của Bác Hồ?\n...',
   'metadata': {'kb_id': 5,
    'subject': 'sd',
    'types': 'general',
    'lesson': 3,
    'chunk_version': 'v3',
    'chunk_order': 1,
    'chunk_id': 'b800ebc7b9d5475d4f395fb2e9b3d345a345c256',
    'heading': ['Khám phá', 'Câu Hỏi'],
    'length': 481,
    'length_token': 243,
    'parent_chunk_id': '8e45f1cb88ed771db310963914ec5f5483a5e91d',
    'subchunk_index': 1,
    'subchunk_total': 3,
    'global_order': 25},
   'score': 1.0},
  